In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
%cd /mlx/users/zongyu.yin/playground/samantha

sample_rate = 24000
hop_length = 240
batch_size = 2
shuffle_buffer_size = 10
num_workers = 2

In [ ]:
import librosa
import matplotlib.pyplot as plt
from librosa.feature.inverse import mel_to_audio
from torchaudio.functional import DB_to_amplitude

def plot_waveform(waveform, sr, title="Waveform", ax=None):
    waveform = waveform.numpy()

    num_channels, num_frames = waveform.shape
    time_axis = torch.arange(0, num_frames) / sr

    if ax is None:
        _, ax = plt.subplots(num_channels, 1)
    ax.plot(time_axis, waveform[0], linewidth=1)
    ax.grid(True)
    ax.set_xlim([0, time_axis[-1]])
    ax.set_title(title)


def plot_spectrogram(specgram, title=None, ylabel="freq_bin", ax=None):
    if ax is None:
        _, ax = plt.subplots(1, 1)
    if title is not None:
        ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.imshow(specgram, origin="lower", aspect="auto", interpolation="nearest")

def mel2audio(mel):
    mel_db2amp = DB_to_amplitude(mel, ref=1, power=1)
    audio = mel_to_audio(mel_db2amp.numpy(), sr=sample_rate, n_fft=512, win_length=400, hop_length=hop_length, n_iter=100)
    return audio
    

# Dataloader

In [ ]:
from recipes.datasets.libritts import LibriTTSWebDataModule

pl_datamodule = LibriTTSWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
    pin_memory=True,
)

train_loader = pl_datamodule.train_dataloader()

In [ ]:
it = iter(train_loader)

In [ ]:
batch = next(it)

In [ ]:
for text, audio in zip(batch["normalized_text"], batch["audio"]):
    print(text)
    ipd.display(ipd.Audio(audio, rate=sample_rate))

# Without VQ

## Load ckpt and model

In [ ]:
import os
import samantha.utils.hdfs_helper as hh
from recipes.umm_062.modules.lit_module import BestRQMelCTC
from transformers import BertTokenizer

ckpt_path = "hdfs://haruna/home/byte_speech_sv/zongyu.yin/logs/umm/finetune_speech_melrecon_ctc_25hz/checkpoints/step=020000-tr_loss=0.2085-val_loss_0=0.8958.ckpt"
cache_dir = ".module_cache/umm"
local_path = f"{cache_dir}/{os.path.basename(ckpt_path)}"
if not os.path.exists(local_path):
    if not hh.get(ckpt_path, local_path):
        raise ConnectionError(f"Cannot retrieve file from {ckpt_path}.")
else:
    print("Cached ckpt found.")
bestrq_mel_ctc = BestRQMelCTC.load_from_checkpoint(local_path).to("cuda").eval()
BestRQMelCTC.tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')

## Get melspec

In [ ]:
model_input = {"audio": batch["audio"].to("cuda"), "normalized_text": batch["normalized_text"]}
recon_mel, gt_mel = bestrq_mel_ctc.get_mel(model_input)
mean = bestrq_mel_ctc.model.model_input_transform.mean
std = bestrq_mel_ctc.model.model_input_transform.std
recon_mel = (recon_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
gt_mel = (gt_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()

## Get audio and plot

In [ ]:
for audio, recon_m, gt_m in zip(batch["audio"], recon_mel, gt_mel):
    print("-----------------------")
    recon_audio = mel2audio(recon_m)
    gt_audio = mel2audio(gt_m)
    fig, axs = plt.subplots(3, 1)
    print("Groud Truth")
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("Groud Truth (mel -> audio)")
    ipd.display(ipd.Audio(gt_audio, rate=sample_rate))
    print("Reconstruction (mel -> audio)")
    ipd.display(ipd.Audio(recon_audio, rate=sample_rate))
    plot_waveform(audio, sample_rate, title="Original waveform", ax=axs[0])
    plot_spectrogram(gt_m, title="GT", ax=axs[1])
    plot_spectrogram(recon_m, title="Recon", ax=axs[2])
    fig.tight_layout()

## CTC

In [ ]:
from torchaudio.models.decoder import ctc_decoder

vocab = list(BestRQMelCTC.tokenizer.get_vocab().keys())
beam_search_decoder = ctc_decoder(
    lexicon=None,
    tokens=vocab,
    nbest=1,
    beam_size=1500,
)

In [ ]:
batch = next(it)
model_input = {"audio": batch["audio"].to("cuda"), "normalized_text": batch["normalized_text"]}

In [ ]:
feature, text_ids = bestrq_mel_ctc.prepare_feature(model_input)
emission, recon_feature = bestrq_mel_ctc.model(feature)
actual_transcript = batch["normalized_text"]

## Greedy

In [ ]:
class GreedyCTCDecoder(torch.nn.Module):
    def __init__(self, labels, blank=0):
        super().__init__()
        self.labels = labels
        self.blank = blank

    def forward(self, emission_batch):
        """Given a sequence emission over labels, get the best path
        Args:
          emission_batch (Tensor): Logit tensors. Shape `[batch, num_seq, num_label]`.

        Returns:
          List[str]: The resulting transcript
        """
        res = []
        for emission in emission_batch:
            indices = torch.argmax(emission, dim=-1)  # [num_seq,]
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            joined = " ".join([self.labels[i] for i in indices])
            res.append(joined.replace("|", " ").strip())
        return res

greedy_decoder = GreedyCTCDecoder(vocab)

In [ ]:
greedy_transcript = greedy_decoder(emission)
for a, g in zip(actual_transcript, greedy_transcript):
    greedy_wer = torchaudio.functional.edit_distance(a.lower(), g.lower()) / len(a)
    print(f"Actual transcript: {a.lower()}")
    print(f"Greedy transcript: {g.lower()}")
    print(f"WER: {greedy_wer}")

In [ ]:
from tqdm import tqdm
from recipes.datasets.libritts import LibriTTSWebDataModule

pl_datamodule = LibriTTSWebDataModule(
    sample_rate=sample_rate,
    batch_size=20,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
    pin_memory=True,
)

it = iter(pl_datamodule.val_dataloader())
for batch in tqdm(it):
    mean_wer = []
    model_input = {"audio": batch["audio"].to("cuda"), "normalized_text": batch["normalized_text"]}
    feature, text_ids = bestrq_mel_ctc.prepare_feature(model_input)
    emission, recon_feature = bestrq_mel_ctc.model(feature)
    actual_transcript = batch["normalized_text"]
    greedy_transcript = greedy_decoder(emission)
    for a, g in zip(actual_transcript, greedy_transcript):
        greedy_wer = torchaudio.functional.edit_distance(a.lower(), g.lower()) / len(a)
        # print(f"Actual transcript: {a.lower()}")
        # print(f"Greedy transcript: {g.lower()}")
        # print(f"WER: {greedy_wer}")
        mean_wer.append(greedy_wer)
    mean_wer = sum(mean_wer) / len(mean_wer)
print(f"Mean WER: {mean_wer}")

## Beam search [WIP]